# YZV202E Optimization for Data Science — Homework 3
**Name:** Yusuf Oğuz  
**Student ID:** 150220322  
**Parameters:** a=5, b=3, c=2, d=2 (Question 1) | a=2 (Question 3, last digit of student ID)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

---
## Question 1: Transportation Problem

**Parameters from student ID 150220322:**
- a = 4th from last digit = 0 → **5** (since 0 → 5)
- b = 3rd from last digit = 3 → **3**
- c = 2nd from last digit = 2 → **2**
- d = last digit = 2 → **2**

**Unit costs (×1000 TL):**
- Ankara → Çanakkale: a = 5
- Ankara → Denizli: b = 3
- Bursa → Çanakkale: c = 2
- Bursa → Denizli: d = 2

**Supply:** Ankara = 90, Bursa = 80  
**Demand:** Çanakkale = 55, Denizli = 65

### 1.1 LP Formulation

**Decision variables:**
- $x_{11}$: units sent from Ankara to Çanakkale
- $x_{12}$: units sent from Ankara to Denizli
- $x_{21}$: units sent from Bursa to Çanakkale
- $x_{22}$: units sent from Bursa to Denizli

**Objective (minimize total cost in ×1000 TL):**
$$\min \; 5x_{11} + 3x_{12} + 2x_{21} + 2x_{22}$$

**Subject to:**
$$x_{11} + x_{12} \leq 90 \quad \text{(Ankara supply)}$$
$$x_{21} + x_{22} \leq 80 \quad \text{(Bursa supply)}$$
$$x_{11} + x_{21} = 55 \quad \text{(Çanakkale demand)}$$
$$x_{12} + x_{22} = 65 \quad \text{(Denizli demand)}$$
$$x_{11}, x_{12}, x_{21}, x_{22} \geq 0$$

### 1.2 Standard Form

Introduce slack variables $s_1, s_2 \geq 0$ for the supply inequalities:

$$\min \; 5x_{11} + 3x_{12} + 2x_{21} + 2x_{22} + 0s_1 + 0s_2$$

**Subject to:**
$$x_{11} + x_{12} + s_1 = 90$$
$$x_{21} + x_{22} + s_2 = 80$$
$$x_{11} + x_{21} = 55$$
$$x_{12} + x_{22} = 65$$
$$x_{11}, x_{12}, x_{21}, x_{22}, s_1, s_2 \geq 0$$

In matrix form $\mathbf{Ax = b}$, $\mathbf{x} \geq 0$:

$$\mathbf{A} = \begin{bmatrix} 1 & 1 & 0 & 0 & 1 & 0 \\ 0 & 0 & 1 & 1 & 0 & 1 \\ 1 & 0 & 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 1 & 0 & 0 \end{bmatrix}, \quad \mathbf{b} = \begin{bmatrix}90\\80\\55\\65\end{bmatrix}, \quad \mathbf{c} = \begin{bmatrix}5\\3\\2\\2\\0\\0\end{bmatrix}$$

### 1.3 Analytical Solution (Northwest Corner + MODI / inspection)

We use the **Minimum Cost Method** (greedy by unit cost):

| Route | Cost | Priority |
|---|---|---|
| Bursa→Çanakkale | 2 | 1st |
| Bursa→Denizli | 2 | 1st (tie) |
| Ankara→Denizli | 3 | 3rd |
| Ankara→Çanakkale | 5 | 4th |

**Step-by-step allocation:**

1. Bursa→Çanakkale: min(80, 55) = 55 → $x_{21}=55$, Çanakkale satisfied, Bursa remaining = 25
2. Bursa→Denizli: min(25, 65) = 25 → $x_{22}=25$, Bursa exhausted, Denizli remaining = 40
3. Ankara→Denizli: min(90, 40) = 40 → $x_{12}=40$, Denizli satisfied, Ankara remaining = 50
4. Ankara→Çanakkale: $x_{11}=0$ (Çanakkale already satisfied)

**Solution:** $x_{11}=0,\; x_{12}=40,\; x_{21}=55,\; x_{22}=25$

**Total cost:** $5(0) + 3(40) + 2(55) + 2(25) = 0 + 120 + 110 + 50 = 280$ (×1000 TL) = **280,000 TL**

**Optimality check:** This is optimal because all demand is met at minimum cost routes. Shifting any flow to Ankara→Çanakkale (cost 5) from Bursa routes (cost 2) would only increase cost.

In [ ]:
# ── Question 1.4: CVXPY Solution ──────────────────────────────────────────────
a, b, c, d = 5, 3, 2, 2
supply = [90, 80]
demand = [55, 65]
cost = np.array([[a, b], [c, d]])

x = cp.Variable((2, 2), nonneg=True)

objective = cp.Minimize(cp.sum(cp.multiply(cost, x)))
constraints = [
    cp.sum(x[0, :]) <= supply[0],  # Ankara supply
    cp.sum(x[1, :]) <= supply[1],  # Bursa supply
    cp.sum(x[:, 0]) == demand[0],  # Çanakkale demand
    cp.sum(x[:, 1]) == demand[1],  # Denizli demand
]

prob = cp.Problem(objective, constraints)
prob.solve()

print("=== CVXPY Solution ===")
print(f"Status: {prob.status}")
print(f"\nFlow matrix (rows=Ankara,Bursa ; cols=Çanakkale,Denizli):")
print(np.round(x.value, 4))
print(f"\nOptimal total cost: {prob.value:.2f} (×1000 TL) = {prob.value*1000:.0f} TL")
print(f"\nAnalytical solution matches CVXPY: {np.isclose(prob.value, 280)}")

In [ ]:
# ── Question 1.5: Effect of a on optimal cost ─────────────────────────────────
a_values = list(range(1, 11))
optimal_costs = []
x11_vals, x12_vals, x21_vals, x22_vals = [], [], [], []

for a_val in a_values:
    cost_mat = np.array([[a_val, b], [c, d]])
    x_var = cp.Variable((2, 2), nonneg=True)
    obj = cp.Minimize(cp.sum(cp.multiply(cost_mat, x_var)))
    cons = [
        cp.sum(x_var[0, :]) <= supply[0],
        cp.sum(x_var[1, :]) <= supply[1],
        cp.sum(x_var[:, 0]) == demand[0],
        cp.sum(x_var[:, 1]) == demand[1],
    ]
    p = cp.Problem(obj, cons)
    p.solve()
    optimal_costs.append(p.value)
    x11_vals.append(round(x_var.value[0, 0]))
    x12_vals.append(round(x_var.value[0, 1]))
    x21_vals.append(round(x_var.value[1, 0]))
    x22_vals.append(round(x_var.value[1, 1]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(a_values, optimal_costs, 'bo-', linewidth=2, markersize=8)
ax1.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='a = b = 3 (tie point)')
ax1.set_xlabel('a (Ankara → Çanakkale unit cost, ×1000 TL)')
ax1.set_ylabel('Optimal Total Cost (×1000 TL)')
ax1.set_title('Effect of a on Optimal Total Cost')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xticks(a_values)

ax2.stackplot(a_values, x11_vals, x12_vals, x21_vals, x22_vals,
              labels=['x11 (Ank→Çan)', 'x12 (Ank→Den)', 'x21 (Bur→Çan)', 'x22 (Bur→Den)'],
              alpha=0.7)
ax2.set_xlabel('a')
ax2.set_ylabel('Flow (units)')
ax2.set_title('Optimal Flow Allocation vs a')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(a_values)

plt.tight_layout()
plt.savefig('q1_a_analysis.png', bbox_inches='tight')
plt.show()

print("\na | Cost  | x11 | x12 | x21 | x22")
print("-" * 40)
for i, a_val in enumerate(a_values):
    print(f"{a_val:2d} | {optimal_costs[i]:5.1f} | {x11_vals[i]:3.0f} | {x12_vals[i]:3.0f} | {x21_vals[i]:3.0f} | {x22_vals[i]:3.0f}")

### 1.5 Discussion: Effect of a

- For **a ≤ 2** (a ≤ c): Ankara→Çanakkale cost ≤ Bursa→Çanakkale, so some flow shifts to Ankara→Çanakkale.
- For **a = 3** (a = b): Ankara→Çanakkale equals Ankara→Denizli cost; the solver may split flows differently but total cost is the same.
- For **a > 3**: The route Ankara→Çanakkale is expensive. Optimal always sends all Çanakkale demand through Bursa (x21=55), and Denizli demand split between Bursa remainder and Ankara. Since Bursa can only supply 80-55=25 to Denizli, Ankara must cover 40 units to Denizli regardless of a.
- **The optimal cost is piecewise linear and non-decreasing in a.** For a ≤ 2 the cost increases with a; for a > 2, the optimal solution avoids the Ankara→Çanakkale route entirely, so the cost becomes **constant** (independent of a).

---
## Question 2: LP Duality

**Primal problem:**
$$\max \; x_1 + x_2$$
$$\text{s.t.} \quad x_1 + 2x_2 \leq 3$$
$$x_1 \geq x_2 \iff -x_1 + x_2 \leq 0$$
$$x_1, x_2 \geq 0$$

### 2.1 Dual Problem Derivation

Write primal in standard form. Associate dual variables $y_1 \geq 0$ with constraint 1 and $y_2 \geq 0$ with constraint 2:

| Primal constraint | Dual variable |
|---|---|
| $x_1 + 2x_2 \leq 3$ | $y_1 \geq 0$ |
| $-x_1 + x_2 \leq 0$ | $y_2 \geq 0$ |

**Dual objective:** minimize $\mathbf{b}^T\mathbf{y} = 3y_1 + 0 \cdot y_2 = 3y_1$

**Dual constraints** (one per primal variable):

For $x_1$: column of A is $[1, -1]^T$, primal cost = 1:
$$y_1 - y_2 \geq 1$$

For $x_2$: column of A is $[2, 1]^T$, primal cost = 1:
$$2y_1 + y_2 \geq 1$$

**Dual problem:**
$$\min \; 3y_1$$
$$\text{s.t.} \quad y_1 - y_2 \geq 1$$
$$2y_1 + y_2 \geq 1$$
$$y_1, y_2 \geq 0$$

**By strong duality**, if both primal and dual are feasible, their optimal values are equal.

In [ ]:
# ── Question 2.2: CVXPY Primal and Dual ───────────────────────────────────────

# Primal
x1_p = cp.Variable(nonneg=True)
x2_p = cp.Variable(nonneg=True)
primal = cp.Problem(
    cp.Maximize(x1_p + x2_p),
    [x1_p + 2*x2_p <= 3, x1_p >= x2_p]
)
primal.solve()

print("=== PRIMAL ===")
print(f"Status: {primal.status}")
print(f"Optimal value: {primal.value:.6f}")
print(f"x1 = {x1_p.value:.6f}, x2 = {x2_p.value:.6f}")

# Dual
y1 = cp.Variable(nonneg=True)
y2 = cp.Variable(nonneg=True)
dual = cp.Problem(
    cp.Minimize(3*y1),
    [y1 - y2 >= 1, 2*y1 + y2 >= 1]
)
dual.solve()

print("\n=== DUAL ===")
print(f"Status: {dual.status}")
print(f"Optimal value: {dual.value:.6f}")
print(f"y1 = {y1.value:.6f}, y2 = {y2.value:.6f}")

print(f"\nStrong duality holds: {np.isclose(primal.value, dual.value)}")

In [ ]:
# ── Question 2.3: Remove x1 >= x2 constraint ──────────────────────────────────
x1_r = cp.Variable(nonneg=True)
x2_r = cp.Variable(nonneg=True)
relaxed = cp.Problem(
    cp.Maximize(x1_r + x2_r),
    [x1_r + 2*x2_r <= 3]
)
relaxed.solve()

print("=== Without x1 >= x2 constraint ===")
print(f"Status: {relaxed.status}")
print(f"Optimal value: {relaxed.value:.6f}")
print(f"x1 = {x1_r.value:.6f}, x2 = {x2_r.value:.6f}")

print(f"\nOriginal optimal: {primal.value:.6f}")
print(f"Relaxed optimal:  {relaxed.value:.6f}")
print(f"Optimal value changed: {not np.isclose(primal.value, relaxed.value)}")
print(f"Optimal solution changed: x1={x1_r.value:.4f} vs {x1_p.value:.4f}, x2={x2_r.value:.4f} vs {x2_p.value:.4f}")

### 2.3 Discussion

**With the constraint $x_1 \geq x_2$:**

Vertices of the feasible region:
- $(0, 0)$: satisfies $x_1 \geq x_2$ ✓, $f = 0$
- $(3, 0)$: satisfies $x_1 + 2x_2 = 3$ ✓, $x_1 \geq x_2$ ✓, $f = 3$
- $(1, 1)$: satisfies $x_1 + 2x_2 = 3$ ✓, $x_1 = x_2$ ✓, $f = 2$

Optimal: $x_1^* = 3, x_2^* = 0$, value = **3**.

**After removing the $x_1 \geq x_2$ constraint:**

New vertices: $(0,0)$, $(3,0)$, $(0, 1.5)$.
- $(3, 0)$: $f = 3$
- $(0, 1.5)$: $f = 1.5$

Optimal remains: $x_1^* = 3, x_2^* = 0$, value = **3**.

**Conclusion:** Removing $x_1 \geq x_2$ does **not** change the optimal value or solution.
This is because the constraint was **inactive** at the original optimum ($x_1 = 3 > 0 = x_2$).
By complementary slackness, an inactive constraint has zero dual variable, meaning it contributes no cost to the objective — removing it leaves the optimum unchanged.
This is consistent with strong duality: the dual variable for an inactive primal constraint is always zero.

---
## Question 3: Lagrange Multipliers

$$\min_{x,y} \; xy \quad \text{s.t.} \quad x^2 + y^2 = a = 2$$

(a = last digit of student ID = 2)

### 3.2 Analytical Solution via Lagrange Multipliers

Form the Lagrangian:
$$\mathcal{L}(x, y, \lambda) = xy + \lambda(x^2 + y^2 - 2)$$

**First-order conditions (stationarity):**
$$\frac{\partial \mathcal{L}}{\partial x} = y + 2\lambda x = 0 \implies y = -2\lambda x \tag{1}$$
$$\frac{\partial \mathcal{L}}{\partial y} = x + 2\lambda y = 0 \implies x = -2\lambda y \tag{2}$$
$$\frac{\partial \mathcal{L}}{\partial \lambda} = x^2 + y^2 - 2 = 0 \tag{3}$$

**Solving:**
Substitute (1) into (2):
$$x = -2\lambda(-2\lambda x) = 4\lambda^2 x$$

Since $x \neq 0$ (otherwise $y=0$ and $x^2+y^2=0\neq 2$):
$$4\lambda^2 = 1 \implies \lambda = \pm \frac{1}{2}$$

**Case 1: $\lambda = -\frac{1}{2}$**
$$y = -2(-\frac{1}{2})x = x \implies y = x$$
From constraint: $2x^2 = 2 \implies x = \pm 1$
Points: $(1,1)$ and $(-1,-1)$, $f = xy = 1$

**Case 2: $\lambda = +\frac{1}{2}$**
$$y = -2(\frac{1}{2})x = -x \implies y = -x$$
From constraint: $2x^2 = 2 \implies x = \pm 1$
Points: $(1,-1)$ and $(-1,1)$, $f = xy = -1$

**Conclusion:**
- **Minimum:** $f^* = -1$ at $(1,-1)$ and $(-1,1)$
- **Maximum:** $f^* = 1$ at $(1,1)$ and $(-1,-1)$

In [ ]:
# ── Question 3.1: Contour Plot ────────────────────────────────────────────────
a_q3 = 2
r = np.sqrt(a_q3)

fig, ax = plt.subplots(1, 1, figsize=(8, 8))

xg = np.linspace(-2, 2, 400)
yg = np.linspace(-2, 2, 400)
X, Y = np.meshgrid(xg, yg)
Z = X * Y

levels = np.linspace(-2, 2, 17)
cs = ax.contour(X, Y, Z, levels=levels, cmap='RdBu_r', alpha=0.7)
ax.clabel(cs, inline=True, fontsize=8, fmt='%.1f')

theta = np.linspace(0, 2*np.pi, 300)
ax.plot(r*np.cos(theta), r*np.sin(theta), 'k-', linewidth=2.5, label=f'$x^2+y^2={a_q3}$ (constraint)')

min_pts = [(1, -1), (-1, 1)]
max_pts = [(1, 1), (-1, -1)]

for pt in min_pts:
    ax.plot(*pt, 'rv', markersize=14, zorder=5)
ax.plot([], [], 'rv', markersize=12, label='Minimum (xy = -1)')

for pt in max_pts:
    ax.plot(*pt, 'g^', markersize=14, zorder=5)
ax.plot([], [], 'g^', markersize=12, label='Maximum (xy = +1)')

# Gradient of objective at optimal points
def grad_f(x, y): return np.array([y, x])
def grad_g(x, y): return np.array([2*x, 2*y])

scale = 0.35
for pt in min_pts + max_pts:
    gf = grad_f(*pt)
    gg = grad_g(*pt)
    ax.annotate('', xy=(pt[0]+scale*gf[0], pt[1]+scale*gf[1]), xytext=pt,
                arrowprops=dict(arrowstyle='->', color='blue', lw=2))
    ax.annotate('', xy=(pt[0]+scale*gg[0], pt[1]+scale*gg[1]), xytext=pt,
                arrowprops=dict(arrowstyle='->', color='darkorange', lw=2))

ax.annotate('', xy=(0, 0), xytext=(0, 0),  # dummy for legend
            arrowprops=dict(arrowstyle='->', color='blue', lw=2))
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='k', lw=2.5, label=f'Constraint: $x^2+y^2={a_q3}$'),
    Line2D([0], [0], marker='v', color='r', markersize=12, label='Minimum (xy = −1)', linestyle='None'),
    Line2D([0], [0], marker='^', color='g', markersize=12, label='Maximum (xy = +1)', linestyle='None'),
    Line2D([0], [0], color='blue', lw=2, label='∇f (objective gradient)'),
    Line2D([0], [0], color='darkorange', lw=2, label='∇g (constraint gradient)'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(f'Contour lines of f(x,y)=xy with constraint $x^2+y^2={a_q3}$')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('q3_contour.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Question 3.3: Effect of a ─────────────────────────────────────────────────
print("Effect of a on optimal solution:")
print("  min xy s.t. x²+y²=a")
print("  Optimal points: (√(a/2), -√(a/2)) and (-√(a/2), √(a/2))")
print("  Minimum value: xy = -√(a/2)·√(a/2) = -a/2\n")
for a_val in [1, 2, 4, 9, 16]:
    min_val = -a_val / 2
    pt = (np.sqrt(a_val/2), -np.sqrt(a_val/2))
    print(f"  a={a_val:2d}: min f* = {min_val:.2f}, at (±{pt[0]:.3f}, ∓{pt[1]:.3f})")

In [ ]:
# ── Question 3.4: scipy verification ─────────────────────────────────────────
from scipy.optimize import minimize as sp_minimize

a_q3 = 2

def objective(v): return v[0] * v[1]
def constraint_eq(v): return v[0]**2 + v[1]**2 - a_q3

results = []
for x0 in [(1.0, -0.5), (-1.0, 0.5), (0.5, 1.0), (-0.5, -1.0)]:
    res = sp_minimize(
        objective, x0,
        method='SLSQP',
        constraints={'type': 'eq', 'fun': constraint_eq},
        options={'ftol': 1e-12, 'maxiter': 1000}
    )
    results.append(res)
    print(f"x0={x0}: f*={res.fun:.6f}, x={res.x[0]:.6f}, y={res.x[1]:.6f}, success={res.success}")

best = min(results, key=lambda r: r.fun)
print(f"\nBest minimum: f* = {best.fun:.6f} at x={best.x[0]:.6f}, y={best.x[1]:.6f}")
print(f"Analytical answer: f* = -1.0")
print(f"Match: {np.isclose(best.fun, -1.0)}")

### 3.3 How a affects the solution

For the general problem $\min xy$ s.t. $x^2 + y^2 = a$:
- The constraint is a circle of radius $\sqrt{a}$.
- Optimal points lie at $(\sqrt{a/2}, -\sqrt{a/2})$ and $(-\sqrt{a/2}, \sqrt{a/2})$.
- Minimum value: $f^* = -a/2$.
- As $a$ increases, the circle grows, and the minimum becomes more negative (larger in magnitude).
- The **direction** of optimal points is always along the $y = -x$ line; only the **scale** changes with $a$.

### 3.5 Maximization vs Minimization

- **Minimization:** $(1,-1)$ and $(-1,1)$, $f^* = -1$ (along $y=-x$ diagonal)
- **Maximization:** $(1,1)$ and $(-1,-1)$, $f^* = +1$ (along $y=x$ diagonal)
- Between the min and max points (on the circle), the function $xy$ transitions from $-1$ to $+1$. The function behavior along the circle is $xy = \frac{a}{2}\cos(2\theta)$ where the point is $(\sqrt{a}\cos\theta, \sqrt{a}\sin\theta)$, oscillating between $\pm a/2$.

### 3.6 SONC, SOSC, and Local Minimizer Check

**Second-Order Necessary Conditions (SONC):**

At a constrained minimum, for all $d$ in the tangent space of the constraint ($\nabla g \cdot d = 0$, i.e., $2xd_1 + 2yd_2 = 0$), we need:
$$d^T \nabla^2_{xx} \mathcal{L} \, d \geq 0$$

The bordered Hessian of $\mathcal{L} = xy + \lambda(x^2+y^2-2)$:
$$\nabla^2_{xx}\mathcal{L} = \begin{bmatrix} 2\lambda & 1 \\ 1 & 2\lambda \end{bmatrix}$$

At minimum point $(1,-1)$, $\lambda = +1/2$:
$$\nabla^2_{xx}\mathcal{L} = \begin{bmatrix} 1 & 1 \\ 1 & 1 \end{bmatrix}$$

Tangent space: $\nabla g = [2, -2]$, so $d = t[1,1]$ for $t \in \mathbb{R}$.
$$d^T \nabla^2_{xx}\mathcal{L} \, d = t^2[1,1]\begin{bmatrix}1&1\\1&1\end{bmatrix}\begin{bmatrix}1\\1\end{bmatrix} = t^2 \cdot 4 > 0 \quad \forall t \neq 0$$

**SOSC satisfied** (strictly positive) → the point is a **strict local minimizer**.

By symmetry, $(-1,1)$ also satisfies SOSC.

Since the constraint set is compact (circle) and all stationary points have been found, the global minimum is $f^* = -1$.

---
## Question 4: KKT Conditions

$$\min_{x_1, x_2} \; x_1^2 + x_2^2$$
$$\text{s.t.} \quad g_1: x_1 - 7 \leq 0$$
$$g_2: -x_1 + \frac{x_2^2}{4} + 4 \leq 0$$

In [ ]:
# ── Question 4.1: Contour plot and feasible region ────────────────────────────
fig, ax = plt.subplots(figsize=(9, 8))

x1g = np.linspace(-2, 10, 500)
x2g = np.linspace(-8, 8, 500)
X1, X2 = np.meshgrid(x1g, x2g)
F = X1**2 + X2**2

# Objective contours
obj_levels = [10, 16, 20, 40, 61, 80, 100, 130]
cs = ax.contour(X1, X2, F, levels=obj_levels, cmap='Blues', alpha=0.8)
ax.clabel(cs, inline=True, fontsize=8, fmt='%.0f')

# Feasible region: g1 <= 0 AND g2 <= 0
G1 = X1 - 7
G2 = -X1 + X2**2/4 + 4
feasible = (G1 <= 0) & (G2 <= 0)
ax.contourf(X1, X2, feasible.astype(float), levels=[0.5, 1.5], colors=['lightgreen'], alpha=0.4)

# Constraint boundaries
ax.axvline(x=7, color='red', linewidth=2)
x2_range = np.linspace(-8, 8, 400)
x1_g2 = x2_range**2/4 + 4
ax.plot(x1_g2, x2_range, 'purple', linewidth=2)

# Optimal point
ax.plot(4, 0, 'r*', markersize=18, zorder=10)

ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Feasible Region, Objective Contours, and Optimal Points')
ax.grid(True, alpha=0.3)
ax.set_xlim(-1, 10); ax.set_ylim(-8, 8)

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_q4 = [
    Patch(facecolor='lightgreen', alpha=0.6, label='Feasible region'),
    Line2D([0],[0], color='red', lw=2, label='$g_1: x_1 = 7$'),
    Line2D([0],[0], color='purple', lw=2, label='$g_2: -x_1 + x_2^2/4 + 4 = 0$'),
    Line2D([0],[0], marker='*', color='r', markersize=14, linestyle='None', label='Optimal $(4, 0)$, $f^*=16$'),
]
ax.legend(handles=legend_q4, loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('q4_contour.png', bbox_inches='tight')
plt.show()

### 4.2 KKT Conditions

**Objective:** $f(x_1, x_2) = x_1^2 + x_2^2$

**Constraints:** $g_1(x_1, x_2) = x_1 - 7 \leq 0$, $\quad g_2(x_1, x_2) = -x_1 + \frac{x_2^2}{4} + 4 \leq 0$

**KKT Lagrangian:**
$$\mathcal{L} = x_1^2 + x_2^2 + \mu_1(x_1 - 7) + \mu_2\left(-x_1 + \frac{x_2^2}{4} + 4\right)$$

**Stationarity conditions:**
$$\frac{\partial \mathcal{L}}{\partial x_1} = 2x_1 + \mu_1 - \mu_2 = 0 \tag{K1}$$
$$\frac{\partial \mathcal{L}}{\partial x_2} = 2x_2 + \mu_2 \frac{x_2}{2} = 0 \tag{K2}$$

**Primal feasibility:**
$$x_1 - 7 \leq 0 \tag{K3}$$
$$-x_1 + \frac{x_2^2}{4} + 4 \leq 0 \tag{K4}$$

**Dual feasibility:**
$$\mu_1, \mu_2 \geq 0 \tag{K5}$$

**Complementary slackness:**
$$\mu_1(x_1 - 7) = 0 \tag{K6}$$
$$\mu_2\left(-x_1 + \frac{x_2^2}{4} + 4\right) = 0 \tag{K7}$$

### 4.3 Analytical Solution via KKT

**Regularity check:** We use LICQ (Linear Independence Constraint Qualification). At a point where both constraints are active, $\nabla g_1 = [1, 0]^T$ and $\nabla g_2 = [-1, x_2/2]^T$. These are linearly independent as long as $x_2 \neq 0$.

**Feasibility check for the origin:** $g_2(0,0) = -0 + 0 + 4 = 4 > 0$ → origin is **not feasible**. The constraint $g_2 \leq 0$ requires $x_1 \geq x_2^2/4 + 4 \geq 4$. So $x_1 \geq 4$; the feasible region is entirely in $x_1 \geq 4$.

**Case 1: $g_1$ inactive ($\mu_1 = 0$), $g_2$ active**

From (K2): $x_2(2 + \mu_2/2) = 0$

- Sub-case 1a: $x_2 = 0$
  - $g_2$ active: $-x_1 + 0 + 4 = 0 \Rightarrow x_1 = 4$
  - From (K1): $2(4) + 0 - \mu_2 = 0 \Rightarrow \mu_2 = 8 > 0$ ✓
  - Check $g_1$: $4 - 7 = -3 \leq 0$ ✓
  - **KKT point: $(4, 0)$, $f = 16$**

- Sub-case 1b: $\mu_2 = -4$ → violates $\mu_2 \geq 0$. **Not valid.**

**Case 2: Both $g_1$ and $g_2$ active**

- $g_1$ active: $x_1 = 7$
- $g_2$ active: $-7 + x_2^2/4 + 4 = 0 \Rightarrow x_2^2 = 12 \Rightarrow x_2 = \pm 2\sqrt{3}$
- From (K2): $2x_2 + \mu_2 x_2/2 = 0$. Since $x_2 \neq 0$: $\mu_2 = -4$ → **violates $\mu_2 \geq 0$**. ✗

Wait — let me recheck: $2x_2(1 + \mu_2/4) = 0$. Since $x_2 \neq 0$: $1 + \mu_2/4 = 0 \Rightarrow \mu_2 = -4$. **Invalid.**

Hmm — actually from (K2): $x_2(2 + \mu_2/2) = 0$. For $x_2 \neq 0$: $\mu_2 = -4 < 0$. **Invalid.**

**Case 3: $g_2$ inactive ($\mu_2 = 0$), $g_1$ active**

- $x_1 = 7$
- From (K2): $2x_2 = 0 \Rightarrow x_2 = 0$
- From (K1): $14 + \mu_1 - 0 = 0 \Rightarrow \mu_1 = -14 < 0$ ✗ **Invalid.**

**Summary of valid KKT points:**

The only valid KKT point is $(4, 0)$ with $f^* = 16$.

> **Note:** This is indeed the global minimizer. The geometry confirms it: we need $x_1 \geq x_2^2/4 + 4 \geq 4$, and the circle $x_1^2 + x_2^2 = r^2$ of smallest radius touching the feasible region first contacts the boundary curve $g_2 = 0$ at the point closest to the origin. Setting $x_2 = 0$ gives $x_1 = 4$, distance $= 4$.

In [ ]:
# ── Question 4.4: CVXPY verification ─────────────────────────────────────────
x1_v = cp.Variable()
x2_v = cp.Variable()

q4_prob = cp.Problem(
    cp.Minimize(x1_v**2 + x2_v**2),
    [
        x1_v - 7 <= 0,
        -x1_v + x2_v**2/4 + 4 <= 0
    ]
)
q4_prob.solve(solver=cp.CLARABEL)

print("=== CVXPY Solution for Question 4 ===")
print(f"Status: {q4_prob.status}")
print(f"Optimal value: {q4_prob.value:.6f}")
print(f"x1 = {x1_v.value:.6f}")
print(f"x2 = {x2_v.value:.6f}")
print(f"\nAnalytical: f* = 16 at (4, 0)")
print(f"Match: {np.isclose(q4_prob.value, 16.0, atol=1e-3)}")

x1_sol, x2_sol = x1_v.value, x2_v.value
print(f"\nConstraint activity:")
print(f"  g1 = x1-7 = {x1_sol - 7:.4f}  (inactive, < 0)")
print(f"  g2 = -x1+x2²/4+4 = {-x1_sol + x2_sol**2/4 + 4:.4f}  (active, = 0)")

### 4.1 Discussion: Activity of Constraints

At the optimal point $(4, 0)$:
- $g_1 = 4 - 7 = -3 < 0$ → **inactive** (slack = 3)
- $g_2 = -4 + 0 + 4 = 0$ → **active** (binding)

Only $g_2$ is active at the optimum, meaning the constraint $-x_1 + x_2^2/4 + 4 \leq 0$ is the binding constraint that determines the optimal solution. The constraint $x_1 \leq 7$ is not active because the optimal $x_1 = 4 < 7$.